In [1]:
%reload_ext autoreload
%autoreload 2

from MBN_Res_Constrn import MBN_RC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#put network you are using eg, full, node removed
name='RE'

#put itr as number of iterations you want
itr=1000
#to get and store iterations data
c=800

#to store iteration data
df_itr=pd.DataFrame()   

#to store local order data
df_loc_data = pd.DataFrame()

#Lists to store transition time and time spent in sync state
trans_time=[]
state_time=[]


while c<itr:
    print("Current iteration is",c)
    mbn = MBN_RC(nepochs=40000, 
                 dt=0.05, 
                 lambda_o=2.86, 
                 alpha=0.01,
                 beta=0.002,
                 plot_bifurcation=False)
    
    mbn.run_model()

    #df to store 1 iteration data
    df=pd.DataFrame(mbn.GLOBAL_ORDER_VERBOSE)
    #concatenating each iteration as a column
    df_itr=pd.concat([df_itr,df], axis=1)

    #changing headers to number of iterations
    df_itr.columns=range(0,df_itr.shape[1])
    #storing dataframe in csv file
    df_itr.to_csv("data8.csv")

    #smoothing the dataframe
    df_smooth=df.rolling(window=800, center=True).mean()

    #defining time and timesteps for data
    time_steps = list(range(0,df_itr.shape[0]))   
    time=np.multiply(time_steps,mbn.dt)

    #dropping the NaN values
    df_na = df_smooth.dropna()
    

    #defining time and time steps for df_na data
    #can drop these 2 lines
    time_steps_na = list(range(0,df_na.shape[0])) 
    time_na=np.multiply(time_steps_na,mbn.dt)

    
    data = np.array(df_na)
    #Looping one cycle of finding transition and time spent in sync state
    while True:
        
        index_arr=np.where(data >= 0.4)[0]

        #Finding if iteration has a transition
        if index_arr.size > 0:
            upper_crossing = index_arr[0]
           #finding transition time thresholds     
            l1 = np.where(data >= 0.1)[0]
            l2=np.where(data <= 0.101)[0]
            low_intersection = np.intersect1d(l1, l2)
            #making sure lower thresholds are for 1st transition
            low_upd =low_intersection[low_intersection<upper_crossing]

            lower_crossing=low_upd[-1]
        
            t=(upper_crossing-lower_crossing)*mbn.dt
            trans_time.append(t)
            #trans_df=pd.concat([trans_df, df_na[i]], axis=1)
            #trans_df stores the iterations which have transitions

            #Extracting local order data for above iteration
            df_dum = pd.DataFrame()
            #defining local order data timestep thresholds
        
            #checking where it crossed 0.3
            m_loc=np.where(data >= 0.3)[0][0]
        

        
            lt_loc = m_loc - 3000
            ut_loc = m_loc + 2000

            #extracting data from df_loc
            header_list=list(range(lt_loc,ut_loc))
            #filtering those columns which lie in between 0 to mbn.nepochs
            header_list_f = [x for x in header_list if 0 <= x < mbn.nepochs]
            df_dum = mbn.df_loc[header_list_f]
            #changing column numbers so that they be concated 1 below other
            df_dum.columns = range(0,df_dum.shape[1])
        

            #creating multi-index dataframe
            index=[[c]*426,list(range(0,426))]
            df_dum=df_dum.set_index(index)
            df_loc_data = pd.concat([df_loc_data,df_dum])
        

        
            #checking for another transition
            fwd_data=np.array(data[upper_crossing:])
        

            #upper_crs=np.where(fwd_data >= 0.45)[0][-1]
            check=np.where(fwd_data <= 0.2)[0]

            if check.size > 0:
                ind=np.where(fwd_data <= 0.1)[0]
                if ind.size > 0:

                    lower_crs = ind[0]
                    lower_crs_up = lower_crs+upper_crossing
                    st_time=(lower_crs_up-lower_crossing)*mbn.dt
                    state_time.append(st_time)
                    #creating data for other cycle of transition and state time
                    data= np.array(data[lower_crs_up:])
                #trans_2df=pd.concat([trans_2df, df_na[i]], axis=1)
                else:
                    
                    break
                    

            else:
                #continue
                
                break
                

        else:
                #data_no_trans=pd.concat([data_no_trans, df_na[i]], axis=1)
            
            break
    
    #increasing count by 1
    c+=1
 

    
#define network for which you are calculatingab

df_tt = pd.DataFrame(trans_time,columns=[name])
df_st=  pd.DataFrame(state_time,columns=[name])


print(np.array(trans_time))
print(np.array(state_time))
print(f'The number of iterations with transition is {len(trans_time)}')    
print(f'The number of iterations with more than 1 transition is {len(state_time)}') 


df_tt.to_csv('tt8_1000.csv')
df_st.to_csv('st8_1000.csv')
df_loc_data=df_loc_data.astype(np.float32)
df_loc_data.to_pickle('l1000.bz2',compression='bz2')

Current iteration is 800
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.47035737793194243
Current iteration is 801
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


In [3]:
df_loc_data.dtypes

0       float64
1       float64
2       float64
3       float64
4       float64
         ...   
4995    float64
4996    float64
4997    float64
4998    float64
4999    float64
Length: 5000, dtype: object

In [4]:
df_loc_data.astype(float)

0         1         2         3         4         5         6     \
801 0    0.236646  0.237031  0.237544  0.238196  0.239002  0.239973  0.241123   
    1    0.100154  0.100773  0.101379  0.101972  0.102558  0.103139  0.103719   
    2    0.054829  0.053727  0.052536  0.051264  0.049922  0.048522  0.047077   
    3    0.190140  0.189433  0.188704  0.187954  0.187185  0.186398  0.185594   
    4    0.050162  0.052380  0.054590  0.056788  0.058971  0.061136  0.063280   
...           ...       ...       ...       ...       ...       ...       ...   
997 421  0.337367  0.334878  0.331921  0.328497  0.324609  0.320260  0.315457   
    422  0.188697  0.190779  0.192742  0.194578  0.196279  0.197837  0.199246   
    423  0.215001  0.217972  0.220722  0.223247  0.225547  0.227619  0.229463   
    424  0.130354  0.131077  0.131606  0.131940  0.132078  0.132022  0.131773   
    425  0.170073  0.170180  0.170183  0.170079  0.169862  0.169531  0.169083   

             7         8         9     ...      4990      4991      4992  \
801 0    0.242464  0.244008  0.245767  ...  0.812136  0.810896  0.809651   
    1    0.104304  0.104897  0.105503  ...  0.481534  0.481171  0.480767   
    2    0.045606  0.044130  0.042671  ...  0.603976  0.603091  0.602243   
    3    0.184776  0.183946  0.183107  ...  0.402818  0.404585  0.406267   
    4    0.065399  0.067491  0.069552  ...  0.527316  0.526458  0.525617   
...           ...       ...       ...  ...       ...       ...       ...   
997 421  0.310204  0.304507  0.298374  ...  0.070037  0.074099  0.078624   
    422  0.200498  0.201585  0.202502  ...  0.196664  0.201299  0.205917   
    423  0.231078  0.232466  0.233627  ...  0.082805  0.082566  0.082256   
    424  0.131337  0.130719  0.129928  ...  0.145414  0.146545  0.147686   
    425  0.168517  0.167832  0.167028  ...  0.344348  0.343459  0.342544   

             4993      4994      4995      4996      4997      4998      4999  
801 0    0.808401  0.807145  0.805883  0.804614  0.803338  0.802053  0.800759  
    1    0.480324  0.479844  0.479327  0.478776  0.478192  0.477576  0.476930  
    2    0.601433  0.600664  0.599935  0.599247  0.598602  0.598001  0.597442  
    3    0.407862  0.409368  0.410783  0.412106  0.413334  0.414466  0.415502  
    4    0.524796  0.523994  0.523211  0.522448  0.521705  0.520981  0.520276  
...           ...       ...       ...       ...       ...       ...       ...  
997 421  0.083526  0.088729  0.094167  0.099786  0.105538  0.111382  0.117287  
    422  0.210506  0.215054  0.219549  0.223978  0.228330  0.232592  0.236753  
    423  0.081875  0.081421  0.080890  0.080282  0.079593  0.078822  0.077964  
    424  0.148831  0.149976  0.151116  0.152247  0.153363  0.154460  0.155533  
    425  0.341609  0.340656  0.339689  0.338712  0.337729  0.336745  0.335764  

[51546 rows x 5000 columns]

In [5]:
df_loc_data.dtypes

0       float64
1       float64
2       float64
3       float64
4       float64
         ...   
4995    float64
4996    float64
4997    float64
4998    float64
4999    float64
Length: 5000, dtype: object

In [8]:
df_loc_data=df_loc_data.astype(np.float32)

In [9]:
df_loc_data.dtypes

0       float32
1       float32
2       float32
3       float32
4       float32
         ...   
4995    float32
4996    float32
4997    float32
4998    float32
4999    float32
Length: 5000, dtype: object

In [11]:
df_loc_data

0         1         2         3         4         5         6     \
801 0    0.236646  0.237031  0.237544  0.238196  0.239002  0.239973  0.241123   
    1    0.100154  0.100773  0.101379  0.101972  0.102558  0.103139  0.103719   
    2    0.054829  0.053727  0.052536  0.051264  0.049922  0.048522  0.047077   
    3    0.190140  0.189433  0.188704  0.187954  0.187185  0.186398  0.185594   
    4    0.050162  0.052380  0.054590  0.056788  0.058971  0.061136  0.063280   
...           ...       ...       ...       ...       ...       ...       ...   
997 421  0.337368  0.334878  0.331921  0.328497  0.324609  0.320260  0.315457   
    422  0.188697  0.190779  0.192742  0.194578  0.196279  0.197837  0.199246   
    423  0.215001  0.217972  0.220722  0.223247  0.225547  0.227619  0.229463   
    424  0.130354  0.131077  0.131606  0.131940  0.132078  0.132022  0.131773   
    425  0.170073  0.170180  0.170183  0.170079  0.169862  0.169531  0.169083   

             7         8         9     ...      4990      4991      4992  \
801 0    0.242464  0.244008  0.245767  ...  0.812136  0.810896  0.809651   
    1    0.104304  0.104897  0.105503  ...  0.481534  0.481171  0.480767   
    2    0.045606  0.044130  0.042671  ...  0.603976  0.603091  0.602243   
    3    0.184776  0.183946  0.183107  ...  0.402818  0.404585  0.406267   
    4    0.065399  0.067491  0.069552  ...  0.527316  0.526458  0.525617   
...           ...       ...       ...  ...       ...       ...       ...   
997 421  0.310204  0.304507  0.298374  ...  0.070037  0.074099  0.078624   
    422  0.200498  0.201585  0.202502  ...  0.196664  0.201299  0.205917   
    423  0.231078  0.232466  0.233627  ...  0.082805  0.082566  0.082256   
    424  0.131337  0.130719  0.129928  ...  0.145414  0.146545  0.147686   
    425  0.168517  0.167832  0.167028  ...  0.344348  0.343459  0.342544   

             4993      4994      4995      4996      4997      4998      4999  
801 0    0.808401  0.807145  0.805883  0.804614  0.803338  0.802053  0.800759  
    1    0.480324  0.479844  0.479327  0.478776  0.478192  0.477576  0.476930  
    2    0.601434  0.600664  0.599935  0.599247  0.598602  0.598001  0.597442  
    3    0.407862  0.409368  0.410783  0.412106  0.413334  0.414466  0.415502  
    4    0.524796  0.523994  0.523211  0.522448  0.521705  0.520981  0.520276  
...           ...       ...       ...       ...       ...       ...       ...  
997 421  0.083526  0.088729  0.094167  0.099786  0.105538  0.111382  0.117287  
    422  0.210506  0.215054  0.219549  0.223978  0.228330  0.232592  0.236753  
    423  0.081875  0.081421  0.080890  0.080282  0.079593  0.078822  0.077964  
    424  0.148831  0.149976  0.151116  0.152247  0.153363  0.154460  0.155533  
    425  0.341609  0.340656  0.339689  0.338712  0.337729  0.336745  0.335764  

[51546 rows x 5000 columns]